In [6]:
import matplotlib.pyplot as plt
import numpy as np
from dataclasses import replace
from scipy.ndimage import gaussian_filter1d
from astropy.stats import biweight_location
from astropy.table import Table, hstack, vstack

import run_config
import utils_lya_halo

from run_config import cfg, smoke
from utils_lya_halo import (run_extract, run_stack, measure, plotting, core, stack, multicat, analysis,
validation)
from utils_lya_halo import guide, pipeline_map, check_guide
from utils_lya_halo.io import read_galaxy_fits, apply_finite_cut

In [2]:
guide()

╔══ LYAHALO · FIELD GUIDE ═══════════════════════════════════════════════════╗
║                                                                            ║
║   pick a section, e.g.  guide("measure")                                   ║
║                                                                            ║
║   ► 1  EXTRACTION (17)  Stage 1-2: turn the fiber data + catalog into the  ║
║   ► 2  MEASURE    (20)  Stage 3: centroids, integrated flux, asymmetry, an ║
║   ► 3  PLOTTING   (25)  Figures for stacks, radial profiles, and per-run d ║
║   ► 4  VALIDATION (30)  Prove the centroid is real: nulls, robustness swee ║
║   ► 5  SAMPLE     (18)  Slice, split, and match the sample -- pure Stage-2 ║
║   ► 6  MISC       (28)  Science extras and separate tracks: virial convers ║
║                                                                            ║
║   detail: guide("run_stack")   search: guide(search="flux")                ║
║   figure: pipeline_map()        audit:  check_guid

In [3]:
guide('MISC')

╔══ MISC ════════════════════════════════════════════════════════════════════╗
║   Science extras and separate tracks: virial conversions, LSF/intrinsic    ║
║   subsections: Virial conversions · LSF / intrinsic profile · Star-PSF &   ║
╚════════════════════════════════════════════════════════════════════════════╝
── Virial conversions ──────────────────────────────────────────────────────
  virial_to_kpc_bins
      Convert R/Rvir bin edges to kpc for ONE galaxy's mass & z
      (Moster+2013 -> R200c). The per-galaxy mapping extraction uses; call
      it to see a galaxy's physical bin edges.
  virial_to_angular_bins
      Convert R/Rvir bin edges to arcsec for one galaxy's mass & z. Useful
      for overlaying bins on imaging.
  physical_kpc_to_arcsec
      Angular size (arcsec) of a physical size (kpc) at redshift z
      (Planck18). The basic cosmology conversion for annotating plots.
  estimate_M200c_R200c_from_Mstar
      Stellar mass -> (M200c, R200c) via Moster+2013 + R200c(z). 

In [5]:
guide(search="centroid")     # grep names + purposes across everything

╔══ SEARCH · 'centroid'  (31 hits) ══════════════════════════════════════════╗
╚════════════════════════════════════════════════════════════════════════════╝
  run_measure  [measure/Drivers]
      Stage 3: bootstrap the Lya centroid (+ blue/red side ratio) and the
      per-pixel stack error from the Stage-2 cube. Fast; requires stacks
      built with keep_cube=True.
  measure_all_bins  [measure/Drivers]
      The config-driven measurement engine run_measure calls: centroid +
      side-ratio bootstrap and stack error for every radial bin. Use
      directly to override stack_method or skip the stack error.
  measure_centroid  [measure/Centroid estimators]
      One entry point for every centroid estimator (dispatch by name), all
      sharing the same return contract. Use this rather than calling a
      specific estimator unless you need that estimator's extra kwargs.
  flux_weighted_centroid  [measure/Centroid estimators]
      Continuum-subtract then flux-weight to a centroid over

In [14]:
guide('run_extract')

┌─ run_extract ────────────────────────────────────── EXTRACTION / Drivers ┐
│ purpose:  Stage 1: extract every galaxy's binned, background-subtracted
│ spectra and write the galaxy FITS. Slow and I/O-bound -- run it once per
│ config; results are cached per galaxy so re-runs are cheap.
│ module:   utils_lya_halo.pipeline
│ example:
│     path = run_extract(cfg)
└────────────────────────────────────────────────────────────────────────────┘


In [13]:
guide('run_stack')

┌─ run_stack ──────────────────────────────────────── EXTRACTION / Drivers ┐
│ purpose:  Stage 2: load the galaxy FITS and coadd into rest-frame stacks
│ (one per combine method). Fast -- re-run freely. keep_cube=True is
│ required if you will measure or bootstrap afterwards.
│ module:   utils_lya_halo.pipeline
│ example:
│     stacks = run_stack(cfg, path, keep_cube=True)
└────────────────────────────────────────────────────────────────────────────┘


In [11]:
guide('run_measure')

┌─ run_measure ───────────────────────────────────────── MEASURE / Drivers ┐
│ purpose:  Stage 3: bootstrap the Lya centroid (+ blue/red side ratio)
│ and the per-pixel stack error from the Stage-2 cube. Fast; requires
│ stacks built with keep_cube=True.
│ module:   utils_lya_halo.pipeline
│ example:
│     boot = run_measure(cfg, stacks)
└────────────────────────────────────────────────────────────────────────────┘
